In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from sklearn.model_selection import train_test_split

In [ ]:
print("Files in dataset path:", os.listdir(path))

In [ ]:

class waterdata(Dataset):
    def __init__(self, root_dir, transform=None, img_size=(256, 256)):
        self.transform = transform
        self.img_size = img_size

        self.image_dir = None
        self.mask_dir = None
        # I have lost in the path so much, so i pulled out this move
        for root, dirs, files in os.walk(root_dir):
            if 'images' in dirs:
                self.image_dir = os.path.join(root, 'images')
                if 'masks' in dirs:
                    self.mask_dir = os.path.join(root, 'masks')
                break

        if self.image_dir is None:
             self.image_dir = root_dir

             self.mask_dir = os.path.join(root_dir, 'masks') if os.path.exists(os.path.join(root_dir, 'masks')) else root_dir

        print(f"Found images at: {self.image_dir}")
        print(f"Found masks at: {self.mask_dir}")

        if not os.path.exists(self.image_dir):
             raise FileNotFoundError(f"Could not locate image directory in {root_dir}")

        self.images = sorted([f for f in os.listdir(self.image_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)

        mask_name = img_name.replace(".jpg", ".png").replace(".jpeg", ".png")
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = Image.open(img_path).convert("RGB")
        image = image.resize(self.img_size, Image.BILINEAR)

        mask = Image.open(mask_path).convert("L")
        mask = mask.resize(self.img_size, Image.NEAREST)

        image = np.array(image)
        mask = np.array(mask)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        mask = torch.from_numpy(mask).long()
        mask = remap_mask(mask)

        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        return image, mask

full_dataset = waterdata(path)

train_idx, val_idx = train_test_split(range(len(full_dataset)), test_size=0.2, random_state=42)
train_loader = DataLoader(torch.utils.data.Subset(full_dataset, train_idx), batch_size=8, shuffle=True)
val_loader = DataLoader(torch.utils.data.Subset(full_dataset, val_idx), batch_size=8, shuffle=False)

images, masks = next(iter(train_loader))
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(images[0].permute(1, 2, 0))
plt.title(f"Image {images[0].shape}")
plt.subplot(1, 2, 2)
plt.imshow(masks[0], cmap='jet')
plt.title(f"Mask {masks[0].shape}")
plt.show()

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO Day3_1_2
import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b0",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=1,  # Binary segmentation (1 output channel)
).to(device)

In [ ]:
# TO DO
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device).long()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device).long()

            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()

    return running_loss / len(loader)

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
# TO DO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate_epoch(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

# Plot loss curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

In [ ]:
# TO DO
model.eval()
images, masks = next(iter(val_loader))
images = images.to(device)
with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1).cpu().numpy()

images = images.cpu().numpy()
masks = masks.cpu().numpy()

# Visualize first 3 samples
plt.figure(figsize=(15, 10))
for i in range(3):
    plt.subplot(3, 3, i*3 + 1)
    plt.imshow(images[i].transpose(1, 2, 0))
    plt.title("Image")
    plt.axis('off')

    plt.subplot(3, 3, i*3 + 2)
    plt.imshow(masks[i], cmap='jet', vmin=0, vmax=7)
    plt.title("Ground Truth")
    plt.axis('off')

    plt.subplot(3, 3, i*3 + 3)
    plt.imshow(preds[i], cmap='jet', vmin=0, vmax=7)
    plt.title("Prediction")
    plt.axis('off')

plt.tight_layout()
plt.show()